In [0]:
%py
from pyspark.sql.functions import *

#Creating and Querying a SQL table from a Spark Dataframe. 

df = (spark.read.format('csv')
      .option('header', 'true')
      .option('inferSchema', 'true')
      .load('/Volumes/prodsqldwh/adls/navfiles/nav_customer_dim.csv')
)


In [0]:
display(df)

In [0]:
df.createOrReplaceTempView('nav_cust_dim')
spark.sql('DESCRIBE nav_cust_dim').show()


In [0]:
display(spark.sql(""" 
                  SELECT * 
                  FROM nav_cust_dim 
                  LIMIT 100 
                  """
))

In [0]:
df2 = (df.select('RecordKey', 'CompanyCode', 'Name', 'CustomerNumber')
         .limit(10))
# Se cambia nombre de columna para el ejemplo:
df2 = df2.withColumnRenamed('RecordKey', 'SalesAmount')
df2.createOrReplaceTempView('nav_cust_dim_sample')
display(spark.sql(""" 
          SELECT * 
          FROM nav_cust_dim_sample
          """
))

In [0]:
# Window Functions in Spark SQL

query = """ 
SELECT  SalesAmount,
        Name, 
        CompanyCode,
        CustomerNumber,      
        LEAD(CustomerNumber, 1) OVER (ORDER BY CustomerNumber) AS NextCustomerNum
FROM nav_cust_dim_sample 
"""

display(spark.sql(query))

In [0]:
query = """ 
SELECT  SalesAmount,
        Name, 
        CompanyCode,
        CustomerNumber,      
        LEAD(CustomerNumber, 1) OVER (PARTITION BY CompanyCode ORDER BY CustomerNumber ASC) AS NextCustomerNum, 
        LAG(CustomerNumber, 1) OVER (PARTITION BY CompanyCode ORDER BY CustomerNumber ASC) AS PrevCustomerNum
FROM nav_cust_dim_sample
ORDER BY CompanyCode DESC;  
"""

display(spark.sql(query))

In [0]:
query = """ 
SELECT  SalesAmount,
        Name, 
        CompanyCode,
        CustomerNumber,      
        LEAD(CustomerNumber, 1) OVER (PARTITION BY CompanyCode ORDER BY CustomerNumber ASC) AS NextCustomerNum, 
        LAG(CustomerNumber, 1) OVER (PARTITION BY CompanyCode ORDER BY CustomerNumber ASC) AS PrevCustomerNum, 
        SUM(SalesAmount) OVER (PARTITION BY CompanyCode ORDER BY CustomerNumber ASC) AS RunningTotal
FROM nav_cust_dim_sample
ORDER BY CompanyCode DESC;  
"""

display(spark.sql(query))

In [0]:
# Dot notation in Spark SQL: 

from pyspark.sql import *

df3 = (
df2.select(
        df2.CompanyCode, 
        df2.CustomerNumber, 
        df2.Name, 
        df2.SalesAmount)
        .withColumn('SalesAmount', df2.SalesAmount.cast('decimal(10,2)'))
        .withColumn('NextSalesAmount', lead(df2.SalesAmount, 1).over(Window.partitionBy(df2.CompanyCode).orderBy(df2.CustomerNumber.asc())))
        .withColumn('PrevSalesAmount', lag(df2.SalesAmount, 1).over(Window.partitionBy(df2.CompanyCode).orderBy(df2.CustomerNumber.asc())))
        .withColumn('RunningTotal', sum(df2.SalesAmount).over(Window.partitionBy(df2.CompanyCode).orderBy(df2.CustomerNumber.asc())))
        .withColumn('SalesVar', when(df2.SalesAmount > lag(df2.SalesAmount, 1)
                                                    .over(Window.partitionBy(df2.CompanyCode).orderBy(df2.CustomerNumber.asc())), 'Sales Increase')
                                .when(df2.SalesAmount < lag(df2.SalesAmount, 1)
                                                    .over(Window.partitionBy(df2.CompanyCode).orderBy(df2.CustomerNumber.asc())), 'Sales Decrease')
                                .otherwise('No Sales Change'))
        .withColumn('VarAmount', df2.SalesAmount - col('PrevSalesAmount'))
        .withColumn('VarPct', ((df2.SalesAmount/col('PrevSalesAmount'))-1)*100)
        .withColumn('VarPct', col('VarPct').cast('decimal(10,2)'))

        .orderBy(df2.CompanyCode.asc())
)

df3.show()


In [0]:
# Group By and Agg functions in Spark SQL: 
df3.groupBy(df3.CompanyCode).agg(min(df3.SalesAmount).alias('MinSales'),
                                 avg(df3.SalesAmount).alias('AvgSales'), 
                                 max(df3.SalesAmount).alias('MaxSales'),
                                 min(df3.VarAmount).alias('MinVarAmount'),
                                 avg(df3.VarAmount).alias('AvgVarAmount'), 
                                 max(df3.VarAmount).alias('MaxVarAmount'),
                                 min(df3.VarPct).alias('MinVarPct'),
                                 avg(df3.VarPct).alias('AvgVarPct'), 
                                 max(df3.VarPct).alias('MaxVarPct')
).show()